# Data Cleaning 08 -- TAQ Millisecond Daily

## Input
`Data/Data_Collection/Initial/08_TAQ_Millisecond/firm_daily_taq/` (partitioned parquet by year, 978,371 rows across 2004--2024, 227 PERMNOs, ~205 factor columns)

## Purpose
Cleans daily stock-level microstructure data from WRDS TAQ IID (Intraday Indicators Database). Key concerns addressed: nanosecond timestamp fields only available from ~2015 onwards, oddlot fields only available from ~2014, late-starting factor groups (BATS exchange, ISO orders, retail flow), cross-dataset alignment with PERMNOs missing from TAQ, and NaN rates that improve dramatically over time as TAQ coverage expanded.

## Stage 0: Load & Inspect
- Loads all year partitions and verifies PERMNO coverage against the master list
- Reports shape, date range, unique PERMNOs, and rows per year with approximate stocks-per-day count

## Stage 1: Per-Factor NaN Rates (All Data)
- Computes NaN rate for all ~205 factor columns and classifies into tiers: 0% NaN, <5%, 5--30%, and >=30% (drop candidates)
- Full listing of factors in the >=30% tier and the 5--30% tier with exact NaN counts

## Stage 2: Per-Factor NaN by Year
- Reports average NaN percentage across all kept factors by year, showing the improvement over time (high in 2004, near-zero from ~2015 onwards)
- For factors in the 5--30% tier, prints a compact heatmap of NaN percentage by year to reveal temporal patterns
- Identifies late-starting factors (first year with <50% NaN after 2004)
- Identifies early-stopping factors (last year with <50% NaN before 2024)

## Stage 3: In-Universe NaN Rates
- Merges with `universe_annual` to restrict analysis to observations where the stock was actually in the top-100 that year
- Recomputes per-factor NaN rates for in-universe observations and compares against overall rates
- Confirms no retained factors exceed 30% NaN when restricted to in-universe observations
- Reports average NaN by year for in-universe observations only

## Stage 4: Per-PERMNO NaN Rates (In-Universe)
- Computes average NaN rate per PERMNO using only retained factors and in-universe observations
- Lists the 15 worst PERMNOs
- Cross-dataset alignment: identifies PERMNOs completely missing from TAQ and PERMNOs missing from TAQ in specific years they were in the universe (typically 1--5 per year, likely ADRs or special share classes)

## Stage 5: Duplicate & Coverage Checks
- Duplicate `(permno, date)` check
- Day-of-week distribution

## Stage 7: Clean & Save

### Columns Dropped (13)
- **10 nanosecond timestamp fields** (`ctime_nano`, `otime_nano`, `ttime_open_nano`, `ttime_close_nano`, `ttime_1pm_nano`, `ttime_4pm_nano`, `nbbot_after_open_nano`, `nbbot_before_close_nano`, `nbbot_1pm_nano`, `nbbot_4pm_nano`): all 59.1% NaN. Nanosecond-precision timing fields only available from ~2015 when TAQ migrated to higher-precision timestamps. Not predictive features.
- **3 oddlot fields** (`n_oddlot_trade`, `oddlot_dollar`, `oddlot_vol`): all 49.1% NaN. WRDS oddlot identification was only applied to data from ~2014 onwards.

### 23 Factors in the 5--30% Range (All Late-Starting, Kept)
- **7 BATS exchange factors** (`*_b`): BATS launched 2005, 66% NaN in 2004, <1% by 2020. In-universe 13.3%.
- **3 ISO factors** (`iso_*`): intermarket sweep orders created by Reg NMS in 2007, 100% NaN in 2004--2006. In-universe 15.2%.
- **13 retail flow factors** (`*_retail`): WRDS retail identification not applied pre-2006, 99% NaN in 2004--2005. In-universe 11.5%.
- All three groups are essentially complete from 2008 onwards (<0.5% NaN). Kept because 18 of 21 sample years have full coverage.

### No Rows Dropped
1--5 PERMNOs per year are missing from TAQ entirely (likely ADRs or special share classes). Handled naturally by cross-sectional aggregation -- if a stock has no TAQ data on a given day, it is excluded from that day's market-level microstructure factors.

### No Winsorisation
Applied cross-sectionally in the merge pipeline.

### No Forward-Fill
Stock-level daily data -- cross-sectional aggregation skips NaN naturally.

### Structural NaN Left As-Is
Remaining NaN is from pre-2007 TAQ coverage gaps and a handful of trading halts. Average in-universe NaN drops from 10.5% in 2004 to 0.1% from 2019 onwards.

## Output
`Data/Data_Collection/Cleaned/08_TAQ_Millisecond/taq_daily_clean.parquet` -- 192 factor columns (down from 205), 978,371 rows

In [1]:
# %% [markdown]
# # Data Cleaning: TAQ Millisecond Daily (firm_daily_taq)
#
# Source: Data/Data_Collection/Initial/08_TAQ_Millisecond/firm_daily_taq/ (partitioned by year)
# Output: Data/Data_Collection/Cleaned/08_TAQ_Millisecond/taq_daily_clean.parquet
#
# Daily stock-level microstructure data from WRDS TAQ (IID — Intraday Indicators
# Database). ~205 pre-computed factors covering spreads, volume decomposition,
# order flow, price impact, volatility, and market quality metrics.
#
# Key concerns:
#   - 13 factors with ≥30% NaN (nano timestamps + oddlot) → drop
#   - NaN rates drop near zero after ~2015 (TAQ coverage expanded)
#   - Some PERMNOs in CRSP but missing from TAQ (cross-dataset alignment)
#   - Should measure NaN rates using only in-universe stocks

# %%
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR     = Path('../../../Data/Data_Collection/Initial/08_TAQ_Millisecond/firm_daily_taq/')
MASTER_PATH = Path('../../../Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/universe_master_clean.parquet')
ANNUAL_PATH = Path('../../../Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/universe_annual_clean.parquet')
OUT_DIR     = Path('../../../Data/Data_Collection/Cleaned/08_TAQ_Millisecond')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD & INSPECT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 0: LOAD & INSPECT — TAQ Daily")
print("=" * 90)

df = pd.read_parquet(RAW_DIR)
df['date'] = pd.to_datetime(df['date'])
master = pd.read_parquet(MASTER_PATH)

print(f"\n  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Unique dates: {df['date'].nunique():,}")
print(f"  Unique PERMNOs: {df['permno'].nunique()}")
print(f"  Master PERMNOs: {len(master)}")

# Verify PERMNOs
data_permnos = set(df['permno'].unique())
master_permnos = set(master['permno'])
extra = data_permnos - master_permnos
missing = master_permnos - data_permnos
print(f"\n  PERMNOs in data but NOT in master: {len(extra)}")
print(f"  PERMNOs in master but NOT in data: {len(missing)}")
if missing:
    print(f"    Missing: {sorted(missing)}")

# Identify column types
all_cols = df.columns.tolist()
id_cols = ['permno']
date_cols = ['date']
meta_cols = [c for c in ['year'] if c in all_cols]
factor_cols = [c for c in all_cols if c not in id_cols + date_cols + meta_cols]

print(f"\n  Total factor columns: {len(factor_cols)}")

# Rows per year
print(f"\n--- Rows per year ---")
rows_per_year = df.groupby(df['date'].dt.year).size()
for year, n in rows_per_year.items():
    avg_stocks = n / 252
    print(f"  {year}: {n:>7,d} rows (~{avg_stocks:.0f} stocks/day)")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: PER-FACTOR NaN RATES (ALL DATA)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 1: PER-FACTOR NaN RATES (ALL DATA)")
print("=" * 90)

n_rows = len(df)

col_nan = df[factor_cols].isna().sum()
col_nan_pct = (col_nan / n_rows * 100).round(2)
col_nan_sorted = col_nan_pct.sort_values(ascending=False)

# Summary by tier
tier_0 = col_nan_pct[col_nan_pct == 0]
tier_clean = col_nan_pct[(col_nan_pct > 0) & (col_nan_pct < 5)]
tier_mid = col_nan_pct[(col_nan_pct >= 5) & (col_nan_pct < 30)]
tier_drop = col_nan_pct[col_nan_pct >= 30]

print(f"\n  Factors with   0% NaN: {len(tier_0)}")
print(f"  Factors with  <5% NaN: {len(tier_clean)}")
print(f"  Factors with 5-30% NaN: {len(tier_mid)}")
print(f"  Factors with ≥30% NaN: {len(tier_drop)}  ← DROP")
print(f"  ──────────────────────────")
print(f"  Total factors: {len(factor_cols)}")
print(f"  Factors to KEEP: {len(tier_0) + len(tier_clean) + len(tier_mid)}")
print(f"  Factors to DROP: {len(tier_drop)}")

# Factors to drop
if len(tier_drop) > 0:
    print(f"\n--- Factors to DROP (≥30% NaN): {len(tier_drop)} ---")
    print(f"\n  {'Factor':<35s} {'NaN %':>8s}  {'Count':>10s}")
    print("  " + "-" * 55)
    for col in tier_drop.sort_values(ascending=False).index:
        pct = col_nan_pct[col]
        count = int(col_nan[col])
        print(f"  {col:<35s} {pct:>7.2f}%  {count:>10,d}")

# Factors in 5-30% range
if len(tier_mid) > 0:
    print(f"\n--- Factors with notable NaN (5-30%): {len(tier_mid)} ---")
    print(f"\n  {'Factor':<35s} {'NaN %':>8s}  {'Count':>10s}")
    print("  " + "-" * 55)
    for col in tier_mid.sort_values(ascending=False).index:
        pct = col_nan_pct[col]
        count = int(col_nan[col])
        print(f"  {col:<35s} {pct:>7.2f}%  {count:>10,d}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: PER-FACTOR NaN BY YEAR (5-30% TIER)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 2: PER-FACTOR NaN BY YEAR")
print("=" * 90)

df['_year'] = df['date'].dt.year
years = sorted(df['_year'].unique())

# ── Average NaN across ALL kept factors, by year ─────────────────────────────
keep_factors_all = list(tier_0.index) + list(tier_clean.index) + list(tier_mid.index)

print(f"\n--- Average NaN % across all {len(keep_factors_all)} kept factors, by year ---")
for y in years:
    mask = df['_year'] == y
    year_nan = df.loc[mask, keep_factors_all].isna().mean().mean() * 100
    n_rows_y = mask.sum()
    bar = "█" * int(year_nan)
    print(f"  {y}: {year_nan:>5.1f}%  ({n_rows_y:>6,d} rows)  {bar}")

# ── Year heatmap for 5-30% tier ─────────────────────────────────────────────
mid_factors = list(tier_mid.sort_values(ascending=False).index)

if len(mid_factors) > 0:
    print(f"\n--- NaN % by year for factors in the 5-30% range ---")

    nan_by_year = df.groupby('_year')[mid_factors].apply(
        lambda x: x.isna().mean() * 100
    ).round(1)

    header = f"  {'Factor':<30s}" + "".join(f" {y:>5d}" for y in years)
    print(f"\n{header}")
    print("  " + "-" * (30 + 6 * len(years)))

    for col in mid_factors:
        row = f"  {col:<30s}"
        for y in years:
            pct = nan_by_year.loc[y, col] if y in nan_by_year.index else 0
            if pct >= 90:
                row += f"  {'--':>5s}"
            elif pct >= 30:
                row += f" {pct:>4.0f}%"
            elif pct > 0:
                row += f" {pct:>4.1f}"
            else:
                row += f"  {'·':>5s}"
        print(row)

# ── Late-starting or early-stopping factors ──────────────────────────────────
print(f"\n--- Late-starting factors (first year with <50% NaN) ---")
for col in mid_factors:
    yearly_nan = df.groupby('_year')[col].apply(lambda x: x.isna().mean() * 100)
    first_good = yearly_nan[yearly_nan < 50]
    if len(first_good) > 0 and first_good.index[0] > years[0]:
        print(f"  {col:<30s} starts ~{first_good.index[0]} "
              f"(NaN before: {yearly_nan.loc[:first_good.index[0]-1].mean():.0f}%)")

print(f"\n--- Early-stopping factors (last year with <50% NaN) ---")
for col in mid_factors + list(tier_clean.index):
    yearly_nan = df.groupby('_year')[col].apply(lambda x: x.isna().mean() * 100)
    last_good = yearly_nan[yearly_nan < 50]
    if len(last_good) > 0 and last_good.index[-1] < years[-1]:
        print(f"  {col:<30s} stops ~{last_good.index[-1]} "
              f"(NaN after: {yearly_nan.loc[last_good.index[-1]+1:].mean():.0f}%)")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3: IN-UNIVERSE NaN RATES
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 3: IN-UNIVERSE NaN RATES")
print("=" * 90)

annual = pd.read_parquet(ANNUAL_PATH)

# Define final keep list (excluding ≥30% NaN)
drop_factors = list(tier_drop.index)
keep_factors = [c for c in factor_cols if c not in drop_factors]

print(f"\n  Factors kept: {len(keep_factors)}")
print(f"  Factors dropped: {len(drop_factors)}")

# Filter to in-universe
df_universe = df.merge(
    annual[['permno', 'year']],
    left_on=['permno', '_year'],
    right_on=['permno', 'year'],
    how='inner'
)
print(f"\n  All rows: {len(df):,}")
print(f"  In-universe rows: {len(df_universe):,}")
print(f"  Dropped (out of universe): {len(df) - len(df_universe):,}")

# ── Per-factor NaN: in-universe vs overall ───────────────────────────────────
n_univ = len(df_universe)
univ_nan = df_universe[keep_factors].isna().sum()
univ_nan_pct = (univ_nan / n_univ * 100).round(2)
univ_nan_sorted = univ_nan_pct.sort_values(ascending=False)

overall_nan_pct = (df[keep_factors].isna().sum() / len(df) * 100).round(2)

# Show only factors with >0% in-universe NaN
print(f"\n--- Per-Factor NaN: In-Universe vs Overall (factors with >1% NaN) ---")
print(f"\n  {'Factor':<35s} {'Universe':>10s}  {'Overall':>10s}  {'Diff':>8s}")
print("  " + "-" * 68)
for col in univ_nan_sorted.index:
    u_pct = univ_nan_pct[col]
    o_pct = overall_nan_pct[col]
    diff = u_pct - o_pct
    if u_pct > 1:
        print(f"  {col:<35s} {u_pct:>9.2f}%  {o_pct:>9.2f}%  {diff:>+7.2f}%")

# ── In-universe NaN tiers ────────────────────────────────────────────────────
u_tier_0 = univ_nan_pct[univ_nan_pct == 0]
u_tier_clean = univ_nan_pct[(univ_nan_pct > 0) & (univ_nan_pct < 5)]
u_tier_mid = univ_nan_pct[(univ_nan_pct >= 5) & (univ_nan_pct < 15)]
u_tier_high = univ_nan_pct[(univ_nan_pct >= 15) & (univ_nan_pct < 30)]
u_tier_drop = univ_nan_pct[univ_nan_pct >= 30]

print(f"\n--- In-Universe NaN Tiers (kept factors) ---")
print(f"  0% NaN:       {len(u_tier_0)}")
print(f"  <5% NaN:      {len(u_tier_clean)}")
print(f"  5-15% NaN:    {len(u_tier_mid)}")
print(f"  15-30% NaN:   {len(u_tier_high)}")
print(f"  ≥30% NaN:     {len(u_tier_drop)}  ← should be zero!")

if len(u_tier_drop) > 0:
    print(f"\n  ⚠ Factors ≥30% NaN even in-universe — consider dropping:")
    for col in u_tier_drop.index:
        print(f"    {col:<35s} {u_tier_drop[col]:.2f}%")

# ── Average NaN by year (in-universe only) ───────────────────────────────────
print(f"\n--- Average NaN across kept factors by year (in-universe) ---")
df_universe['_year2'] = df_universe['date'].dt.year
for y in years:
    mask = df_universe['_year2'] == y
    if mask.sum() == 0:
        continue
    year_nan = df_universe.loc[mask, keep_factors].isna().mean().mean() * 100
    n_rows_y = mask.sum()
    bar = "█" * int(year_nan)
    print(f"  {y}: {year_nan:>5.1f}%  ({n_rows_y:>6,d} rows)  {bar}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4: PER-PERMNO NaN RATES (IN-UNIVERSE)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 4: PER-PERMNO NaN RATES (IN-UNIVERSE)")
print("=" * 90)

permno_nan_univ = (
    df_universe.groupby('permno')[keep_factors]
    .apply(lambda x: x.isna().mean().mean() * 100)
    .sort_values(ascending=False)
)

print(f"\n  PERMNOs with <1% avg NaN:    {(permno_nan_univ < 1).sum()}")
print(f"  PERMNOs with 1-5% avg NaN:   {((permno_nan_univ >= 1) & (permno_nan_univ < 5)).sum()}")
print(f"  PERMNOs with 5-15% avg NaN:  {((permno_nan_univ >= 5) & (permno_nan_univ < 15)).sum()}")
print(f"  PERMNOs with >15% avg NaN:   {(permno_nan_univ >= 15).sum()}")

# Worst PERMNOs
worst = permno_nan_univ.head(15)
print(f"\n  15 worst PERMNOs (in-universe, kept factors):")
print(f"  {'PERMNO':>8s}  {'Avg NaN %':>10s}  {'Univ Rows':>10s}  {'Yrs':>5s}")
print("  " + "-" * 40)
for permno, pct in worst.items():
    n_rows_u = len(df_universe[df_universe['permno'] == permno])
    n_yrs = len(annual[annual['permno'] == permno])
    print(f"  {int(permno):>8d}  {pct:>9.2f}%  {n_rows_u:>10,d}  {n_yrs:>5d}")

# ── Cross-dataset alignment: PERMNOs missing from TAQ ────────────────────────
print(f"\n--- PERMNOs in master but missing from TAQ ---")
missing_from_taq = master_permnos - data_permnos
if len(missing_from_taq) > 0:
    print(f"  {len(missing_from_taq)} PERMNOs completely missing:")
    for p in sorted(missing_from_taq):
        n_yrs = len(annual[annual['permno'] == p])
        print(f"    PERMNO {int(p):>6d}  ({n_yrs} yrs in top-100)")
else:
    print(f"  ✓ All master PERMNOs present in TAQ")

# PERMNOs with partial year coverage in TAQ
print(f"\n--- PERMNOs in universe some years but missing from TAQ those years ---")
for y in years:
    universe_permnos_y = set(annual[annual['year'] == y]['permno'])
    taq_permnos_y = set(df[df['_year'] == y]['permno'].unique())
    missing_y = universe_permnos_y - taq_permnos_y
    if missing_y:
        print(f"  {y}: {len(missing_y)} missing — {sorted([int(p) for p in missing_y])}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 5: DUPLICATE & COVERAGE CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 5: DUPLICATE & COVERAGE CHECKS")
print("=" * 90)

# Duplicates
print(f"\n--- Duplicate (permno, date) ---")
n_dupes = df.duplicated(subset=['permno', 'date']).sum()
if n_dupes == 0:
    print(f"  ✓ No duplicates")
else:
    print(f"  ⚠ {n_dupes} duplicates")

# Day-of-week
print(f"\n--- Day-of-week distribution ---")
dow = df['date'].dt.day_name().value_counts()
print(dow.to_string())

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 6: SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 6: SUMMARY")
print("=" * 90)

print(f"""
FACTOR TIERS (overall):
  0% NaN:       {len(tier_0):>4d}
  <5% NaN:      {len(tier_clean):>4d}
  5-30% NaN:    {len(tier_mid):>4d}
  ≥30% NaN:     {len(tier_drop):>4d}  ← DROP
  ────────────────────────
  TOTAL KEEP:   {len(keep_factors):>4d}
  TOTAL DROP:   {len(drop_factors):>4d}

Review the year heatmap and in-universe NaN tiers.
Check if any additional factors should be dropped:
  - Factors that stop before 2024 (early-stopping)
  - Factors that are ≥30% NaN in-universe

Paste back the output and I will write the cleaning cell.
""")

# Clean up
df = df.drop(columns=['_year'], errors='ignore')

STAGE 0: LOAD & INSPECT — TAQ Daily

  Shape: 978,371 rows × 208 columns
  Date range: 2004-01-02 → 2024-12-31
  Unique dates: 5,285
  Unique PERMNOs: 227
  Master PERMNOs: 227

  PERMNOs in data but NOT in master: 0
  PERMNOs in master but NOT in data: 0

  Total factor columns: 205

--- Rows per year ---
  2004:  49,840 rows (~198 stocks/day)
  2005:  49,988 rows (~198 stocks/day)
  2006:  48,750 rows (~193 stocks/day)
  2007:  48,975 rows (~194 stocks/day)
  2008:  49,450 rows (~196 stocks/day)
  2009:  47,814 rows (~190 stocks/day)
  2010:  46,814 rows (~186 stocks/day)
  2011:  47,348 rows (~188 stocks/day)
  2012:  47,188 rows (~187 stocks/day)
  2013:  48,242 rows (~191 stocks/day)
  2014:  47,373 rows (~188 stocks/day)
  2015:  46,396 rows (~184 stocks/day)
  2016:  46,139 rows (~183 stocks/day)
  2017:  45,352 rows (~180 stocks/day)
  2018:  44,826 rows (~178 stocks/day)
  2019:  44,624 rows (~177 stocks/day)
  2020:  44,087 rows (~175 stocks/day)
  2021:  44,060 rows (~175 st

In [2]:
# %% [markdown]
# ## Stage 7: Clean & Save
#
# **Data overview:**
# Daily stock-level microstructure data from WRDS TAQ IID (Intraday Indicators
# Database). 978,371 rows across 2004–2024 for 227 PERMNOs (~175–198 stocks/day).
# Already filtered to universe PERMNOs during collection.
#
# **Columns dropped (13):**
# - 10 nanosecond timestamp fields (`ctime_nano`, `otime_nano`, `ttime_open_nano`,
#   `ttime_close_nano`, `ttime_1pm_nano`, `ttime_4pm_nano`, `nbbot_after_open_nano`,
#   `nbbot_before_close_nano`, `nbbot_1pm_nano`, `nbbot_4pm_nano`): all 59.1% NaN.
#   These are nanosecond-precision timing fields only available from ~2015 onwards
#   when TAQ migrated to higher-precision timestamps. Not predictive features.
# - 3 oddlot fields (`n_oddlot_trade`, `oddlot_dollar`, `oddlot_vol`): all 49.1% NaN.
#   WRDS oddlot identification was only applied to data from ~2014 onwards.
#
# **Factors retained: 192.** In-universe NaN tiers:
#   0% NaN: 127, <5%: 42, 5–15%: 20, 15–30%: 3, ≥30%: 0.
#
# **23 factors in the 5–30% range are all late-starting, not problematic:**
# - 7 BATS exchange factors (`*_b`): BATS launched 2005, 66% NaN in 2004,
#   <1% by 2020. In-universe 13.3%.
# - 3 ISO factors (`iso_*`): intermarket sweep orders created by Reg NMS in
#   2007, 100% NaN in 2004–2006. In-universe 15.2%.
# - 13 retail flow factors (`*_retail`): WRDS retail identification not applied
#   pre-2006, 99% NaN in 2004–2005. In-universe 11.5%.
# All three groups are essentially complete from 2008 onwards (<0.5% NaN).
# Kept because 18 of 21 sample years have full coverage.
#
# **No rows dropped.** 1–5 PERMNOs per year are missing from TAQ entirely
# (likely ADRs or special share classes). These are handled naturally by the
# cross-sectional aggregation — if a stock has no TAQ data on a given day,
# it's excluded from that day's market-level microstructure factors.
#
# **No winsorisation.** Applied cross-sectionally in merge pipeline.
#
# **No forward-fill.** Stock-level daily data — cross-sectional aggregation
# skips NaN naturally.
#
# **Structural NaN left as-is.** Remaining NaN is from pre-2007 TAQ coverage
# gaps and a handful of trading halts. Average in-universe NaN drops from
# 10.5% in 2004 to 0.1% from 2019 onwards.
#
# **Factors retained: 192** (was 205 before cleaning)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 7: CLEAN & SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 7: CLEAN & SAVE")
print("=" * 90)

# ── 7a. Drop columns ────────────────────────────────────────────────────────
drop_cols = list(tier_drop.index)
drop_cols_present = [c for c in drop_cols if c in df.columns]

df = df.drop(columns=drop_cols_present)

meta_cols = [c for c in ['year'] if c in df.columns]
factor_cols_final = [c for c in df.columns if c not in ['date', 'permno'] + meta_cols]

print(f"\n  Dropped {len(drop_cols_present)} columns:")
print(f"    Nano timestamps: {len([c for c in drop_cols_present if 'nano' in c])}")
print(f"    Oddlot fields:   {len([c for c in drop_cols_present if 'oddlot' in c])}")
print(f"  Remaining factor columns: {len(factor_cols_final)}")

# ── 7b. Final NaN report ────────────────────────────────────────────────────
nan_check = df[factor_cols_final].isna().sum()
nan_cols = nan_check[nan_check > 0].sort_values(ascending=False)
total_nan = nan_cols.sum()
total_cells = len(df) * len(factor_cols_final)
print(f"\n  Total NaN: {total_nan:,} / {total_cells:,} ({total_nan/total_cells*100:.2f}%)")
print(f"  Factors with any NaN: {len(nan_cols)} / {len(factor_cols_final)}")

# ── 7c. Final summary ───────────────────────────────────────────────────────
print(f"\n  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  PERMNOs: {df['permno'].nunique()}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

print(f"\n  Sample (first 5 rows, first 8 factors):")
show_cols = ['permno', 'date'] + factor_cols_final[:8]
print(df[show_cols].head(5).to_string(index=False))

# ── 7d. Save ─────────────────────────────────────────────────────────────────
out_path = OUT_DIR / 'taq_daily_clean.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]:,} rows × {df.shape[1]} columns "
      f"({len(factor_cols_final)} factors)")

print("\nCleaning complete.")

STAGE 7: CLEAN & SAVE

  Dropped 13 columns:
    Nano timestamps: 10
    Oddlot fields:   3
  Remaining factor columns: 192

  Total NaN: 3,472,915 / 187,847,232 (1.85%)
  Factors with any NaN: 179 / 192

  Final shape: 978,371 rows × 195 columns
  PERMNOs: 227
  Date range: 2004-01-02 → 2024-12-31

  Sample (first 5 rows, first 8 factors):
 permno       date  avg_buy_price_inst20k  avg_buy_price_inst50k  avg_buy_price_lr  avg_buy_price_retail  avg_buy_price_tick  avg_buy_price_wrds  avg_price_a  avg_price_b
  10104 2004-01-02               13.23915              13.248301         13.237424                  <NA>           13.236214           13.236712    13.172909    13.259308
  10104 2004-01-05              13.413546              13.408727         13.404095                  <NA>            13.40558            13.40505    13.531745    13.328134
  10104 2004-01-06              13.573251              13.568148         13.558652                  <NA>           13.559584           13.559472